In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Building a Drift Monitoring Dashboard\n",
    "\n",
    "You've deployed a model. Now what?\n",
    "\n",
    "This notebook shows how to set up continuous monitoring that answers:\n",
    "1. Is my input data still similar to training data?\n",
    "2. Is my model's behavior changing over time?\n",
    "3. When should I retrain?"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "import sys\n",
    "sys.path.append('..')\n",
    "from drift_detection import KSTest, PSI\n",
    "\n",
    "sns.set_style(\"whitegrid\")\n",
    "np.random.seed(42)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Simulating Production Data\n",
    "\n",
    "We'll simulate 12 weeks where:\n",
    "- Weeks 1-4: Data matches training distribution\n",
    "- Weeks 5-8: Gradual drift begins\n",
    "- Weeks 9-12: Significant drift"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "n_reference = 5000\n",
    "reference_data = {\n",
    "    'feature_1': np.random.normal(0.50, 0.10, n_reference),\n",
    "    'feature_2': np.random.normal(100, 15, n_reference),\n",
    "    'feature_3': np.random.normal(0.30, 0.08, n_reference)\n",
    "}\n",
    "reference_df = pd.DataFrame(reference_data)\n",
    "\n",
    "print(\"Reference distribution statistics:\")\n",
    "reference_df.describe().round(3)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def generate_weekly_data(week, samples_per_week=500):\n",
    "    f1_mean, f1_std = 0.50, 0.10\n",
    "    f2_mean, f2_std = 100, 15\n",
    "    f3_mean, f3_std = 0.30, 0.08\n",
    "    \n",
    "    if week <= 4:\n",
    "        pass  # No drift\n",
    "    elif week <= 8:\n",
    "        drift_factor = (week - 4) / 4\n",
    "        f1_mean += 0.03 * drift_factor\n",
    "        f2_mean += 5 * drift_factor\n",
    "    else:\n",
    "        f1_mean += 0.08\n",
    "        f1_std *= 1.3\n",
    "        f2_mean += 12\n",
    "        f2_std *= 1.2\n",
    "        f3_mean += 0.05\n",
    "    \n",
    "    return pd.DataFrame({\n",
    "        'feature_1': np.random.normal(f1_mean, f1_std, samples_per_week),\n",
    "        'feature_2': np.random.normal(f2_mean, f2_std, samples_per_week),\n",
    "        'feature_3': np.random.normal(f3_mean, f3_std, samples_per_week),\n",
    "        'week': week\n",
    "    })\n",
    "\n",
    "weekly_data = [generate_weekly_data(w) for w in range(1, 13)]\n",
    "production_df = pd.concat(weekly_data, ignore_index=True)\n",
    "\n",
    "print(f\"Generated {len(production_df)} samples across 12 weeks\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Running Weekly Drift Checks"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "ks_detector = KSTest(alpha=0.05)\n",
    "psi_calculator = PSI(n_bins=10)\n",
    "\n",
    "monitoring_results = []\n",
    "features = ['feature_1', 'feature_2', 'feature_3']\n",
    "\n",
    "for week in range(1, 13):\n",
    "    week_data = production_df[production_df['week'] == week]\n",
    "    \n",
    "    for feature in features:\n",
    "        ref_values = reference_df[feature].values\n",
    "        cur_values = week_data[feature].values\n",
    "        \n",
    "        ks_result = ks_detector.detect(ref_values, cur_values)\n",
    "        psi_result = psi_calculator.calculate(ref_values, cur_values)\n",
    "        \n",
    "        monitoring_results.append({\n",
    "            'week': week,\n",
    "            'feature': feature,\n",
    "            'ks_statistic': ks_result.statistic,\n",
    "            'ks_pvalue': ks_result.p_value,\n",
    "            'ks_drift': ks_result.drift_detected,\n",
    "            'psi': psi_result.psi,\n",
    "            'psi_level': psi_result.drift_level\n",
    "        })\n",
    "\n",
    "monitoring_df = pd.DataFrame(monitoring_results)\n",
    "print(f\"Collected {len(monitoring_df)} feature-week observations\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## The Dashboard View"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)\n",
    "\n",
    "ax1 = axes[0]\n",
    "for feature in features:\n",
    "    feature_data = monitoring_df[monitoring_df['feature'] == feature]\n",
    "    ax1.plot(feature_data['week'], feature_data['psi'], 'o-', label=feature, linewidth=2, markersize=8)\n",
    "\n",
    "ax1.axhline(y=0.1, color='orange', linestyle='--', alpha=0.7, label='Warning (0.1)')\n",
    "ax1.axhline(y=0.2, color='red', linestyle='--', alpha=0.7, label='Critical (0.2)')\n",
    "ax1.axvspan(4.5, 8.5, alpha=0.1, color='orange')\n",
    "ax1.axvspan(8.5, 12.5, alpha=0.1, color='red')\n",
    "\n",
    "ax1.set_ylabel('PSI Score', fontsize=12)\n",
    "ax1.set_title('Population Stability Index Over Time', fontsize=14)\n",
    "ax1.legend(loc='upper left', ncol=2)\n",
    "ax1.grid(True, alpha=0.3)\n",
    "ax1.set_ylim(0, 0.35)\n",
    "\n",
    "ax2 = axes[1]\n",
    "for feature in features:\n",
    "    feature_data = monitoring_df[monitoring_df['feature'] == feature]\n",
    "    ax2.semilogy(feature_data['week'], feature_data['ks_pvalue'], 'o-', label=feature, linewidth=2, markersize=8)\n",
    "\n",
    "ax2.axhline(y=0.05, color='red', linestyle='--', alpha=0.7, label='alpha=0.05')\n",
    "ax2.axvspan(4.5, 8.5, alpha=0.1, color='orange')\n",
    "ax2.axvspan(8.5, 12.5, alpha=0.1, color='red')\n",
    "\n",
    "ax2.set_xlabel('Week', fontsize=12)\n",
    "ax2.set_ylabel('KS Test p-value (log)', fontsize=12)\n",
    "ax2.set_title('KS Test Significance Over Time', fontsize=14)\n",
    "ax2.legend(loc='upper right')\n",
    "ax2.grid(True, alpha=0.3)\n",
    "ax2.set_xticks(range(1, 13))\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Weekly Alert Report"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def generate_weekly_report(week, monitoring_df):\n",
    "    week_data = monitoring_df[monitoring_df['week'] == week]\n",
    "    \n",
    "    print(f\"\\n{'='*60}\")\n",
    "    print(f\" DRIFT MONITORING REPORT - WEEK {week}\")\n",
    "    print(f\"{'='*60}\")\n",
    "    \n",
    "    alerts = []\n",
    "    warnings = []\n",
    "    \n",
    "    for _, row in week_data.iterrows():\n",
    "        status = \"STABLE\"\n",
    "        \n",
    "        if row['psi'] > 0.2 or row['ks_drift']:\n",
    "            status = \"DRIFT\"\n",
    "            alerts.append(row['feature'])\n",
    "        elif row['psi'] > 0.1:\n",
    "            status = \"WARNING\"\n",
    "            warnings.append(row['feature'])\n",
    "        \n",
    "        print(f\"\\n[{status}] {row['feature']}\")\n",
    "        print(f\"   PSI: {row['psi']:.4f} ({row['psi_level']})\")\n",
    "        print(f\"   KS p-value: {row['ks_pvalue']:.2e}\")\n",
    "    \n",
    "    print(f\"\\n{'-'*60}\")\n",
    "    print(f\"SUMMARY: {len(alerts)} alerts, {len(warnings)} warnings\")\n",
    "    \n",
    "    if alerts:\n",
    "        print(f\"\\nACTION REQUIRED: Significant drift detected.\")\n",
    "\n",
    "generate_weekly_report(3, monitoring_df)\n",
    "generate_weekly_report(7, monitoring_df)\n",
    "generate_weekly_report(11, monitoring_df)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## When to Retrain?\n",
    "\n",
    "| Week | Status | Action |\n",
    "|------|--------|--------|\n",
    "| 1-4 | Stable | Continue monitoring |\n",
    "| 5-6 | Warning | Investigate root cause |\n",
    "| 7-8 | Mixed alerts | Prepare retraining pipeline |\n",
    "| 9+ | Critical | **Retrain required** |"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\nDRIFT TIMELINE\")\n",
    "print(\"=\"*60)\n",
    "\n",
    "for feature in features:\n",
    "    feature_data = monitoring_df[monitoring_df['feature'] == feature]\n",
    "    \n",
    "    first_warning = feature_data[feature_data['psi'] > 0.1]['week'].min()\n",
    "    first_critical = feature_data[feature_data['psi'] > 0.2]['week'].min()\n",
    "    first_ks_drift = feature_data[feature_data['ks_drift']]['week'].min()\n",
    "    \n",
    "    print(f\"\\n{feature}:\")\n",
    "    print(f\"  First PSI warning (>0.1):  Week {first_warning if pd.notna(first_warning) else 'Never'}\")\n",
    "    print(f\"  First PSI critical (>0.2): Week {first_critical if pd.notna(first_critical) else 'Never'}\")\n",
    "    print(f\"  First KS drift detected:   Week {first_ks_drift if pd.notna(first_ks_drift) else 'Never'}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Key Takeaways\n",
    "\n",
    "1. **Monitor continuously** - Don't wait for complaints\n",
    "2. **Use multiple metrics** - KS catches sudden shifts, PSI tracks gradual drift\n",
    "3. **Set clear thresholds** - Know what triggers investigation vs. action\n",
    "4. **Automate alerts** - This notebook becomes a scheduled job\n",
    "5. **Document decisions** - When you don't retrain, document why"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}